In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!pip install -q kaggle
!kaggle datasets download -d google/google-landmarks-dataset -p /content/gld_dataset --unzip


Dataset URL: https://www.kaggle.com/datasets/google/google-landmarks-dataset
License(s): other
  0% 0.00/1.12k [00:00<?, ?B/s]
100% 1.12k/1.12k [00:00<00:00, 4.72MB/s]


In [ ]:
import pandas as pd

# Load datasets
train_df = pd.read_csv('/content/gld_dataset/train.csv')  # From Kaggle
labels_df = pd.read_csv('/content/train_label_to_category.csv')  # From GitHub



In [ ]:
# Convert 'category' URL into readable landmark name
labels_df['landmark_name'] = labels_df['category'].apply(
    lambda x: x.split('/')[-1].replace('_', ' ').title() if isinstance(x, str) else ''
)


In [ ]:
uae_keywords = ['Dubai', 'Uae', 'Sharjah', 'Abu Dhabi', 'Burj', 'Palm']

# Filter directly on raw category string (don't parse/replace title case)
uae_landmarks = labels_df[
    labels_df['category'].str.contains('|'.join(uae_keywords), case=False, na=False)
]

uae_ids = uae_landmarks['label'].astype(int).tolist()

print("✅ Filtered UAE landmark IDs:", len(uae_ids))
uae_landmarks.head()


KeyError: 'label'

In [ ]:
# Filter the image dataset using landmark IDs
uae_images = train_df[train_df['landmark_id'].isin(uae_ids)]

print("✅ Number of UAE landmark images:", len(uae_images))
uae_images.head()


✅ Number of UAE landmark images: 0


,id,url,landmark_id


In [ ]:
import pandas as pd

# Load training image-to-landmark mappings
train_df = pd.read_csv("/content/train_clean.csv")
label_df = pd.read_csv("/content/train_label_to_category.csv")


In [ ]:
train = pd.read_csv("train_clean.csv")
meta = pd.read_csv("train_label_to_category.csv")


In [ ]:
target_landmarks = ['Burj Khalifa', 'Burj Al Arab', 'Dubai Frame', 'Atlantis The Palm']

matched_ids = label_df[label_df['landmark_name'].isin(target_landmarks)]['label'].astype(int).tolist()

filtered_images = train_df[train_df['landmark_id'].isin(matched_ids)]
filtered_images.head()


,landmark_id,images
55659,138617,10366c9b8a19206f 11ebfbe1b6ccad51 12cc376d0a01...
65418,163459,3407604a11ca6612 4e97a05c03ae2e69 5249e252180e...


In [ ]:
import pandas as pd

# Load label-to-category metadata
label_df = pd.read_csv('/content/train_label_to_category.csv')

# Extract readable landmark names
label_df['landmark_name'] = label_df['category'].apply(
    lambda url: url.split('/')[-1].replace('_', ' ').title()
)

# Filter for UAE / Dubai names
uae_landmarks = label_df[
    label_df['landmark_name'].str.contains('Dubai|Abu Dhabi|Sharjah|Uae|Burj', case=False, na=False)
]

# Show results
print("UAE landmarks found:", uae_landmarks[['landmark_id', 'landmark_name']].drop_duplicates())

UAE landmarks found:         landmark_id                                      landmark_name
6820           6820                         Category:Dubai Sports City
12177         12177                                 Category:Dubai Zoo
13686         13686                  Category:Manuae (Society Islands)
18411         18411                     Category:Bollywood Parks Dubai
18647         18647                            Category:Downtown Dubai
21058         21058                              Category:Dubai Museum
25445         25445                              Category:Sharjah Fort
34456         34456                               Category:Dubai Creek
41811         41811                          Category:Mount Tapuaenuku
42523         42523                                 Category:Tabuaeran
43041         43041                     Category:Shahi Burj (Red Fort)
46452         46452                 Category:Musamman Burj (Agra Fort)
54010         54010                       Category:Expo 

In [ ]:

# Filter only UAE landmarks
uae_ids = uae_landmarks['label'].astype(int).tolist()
uae_images = train[train['landmark_id'].isin(uae_ids)]

print("Number of images available:", len(uae_images))
uae_images.head()

Number of images available: 19


,landmark_id,images
7520,18647,02757fef12d5a399 0f0e1e6cb5c95212 11a49758adc9...
8470,21058,04cf1782c7610b73 0d63f923933982ac 0ff3473b68e8...
10243,25445,3374a3a649010ed9 4b3792c539392197 503400644b6d...
13948,34456,00900ef83ea8b640 02af1871d5ebe2df 0652fe328e3a...
17413,43041,02f13ff70161f092 06146aceb7043903 18b2b4fc2346...


In [ ]:
# Function to extract landmark name from category URL
def extract_landmark_name(category_url):
    if isinstance(category_url, str):
        # Extract the last part of the URL after the last slash and replace underscores with spaces
        return category_url.split('/')[-1].replace('_', ' ')
    return ''

# Extract landmark names into a new column
meta['extracted_name'] = meta['category'].apply(extract_landmark_name)

# Check if target names exist in the 'extracted_name' column
exists = meta[meta['extracted_name'].str.contains('|'.join(target_names), case=False, na=False)]

if not exists.empty:
    print("Found these landmarks in metadata:")
    print(exists)
else:
    print("No target landmarks found in metadata.")

Found these landmarks in metadata:
        landmark_id                                           category  \
80866         80866  http://commons.wikimedia.org/wiki/Category:Atl...   
138617       138617  http://commons.wikimedia.org/wiki/Category:Bur...   
163459       163459  http://commons.wikimedia.org/wiki/Category:Bur...   

                    extracted_name  
80866   Category:Atlantis The Palm  
138617       Category:Burj Al Arab  
163459       Category:Burj Khalifa  


In [ ]:
os.makedirs("gld_templates", exist_ok=True)

def download_images(image_ids, out_dir='gld_templates', limit=10):
    for img_id in tqdm(image_ids[:limit]):
        prefix = img_id[0]
        url = f'https://storage.googleapis.com/landmark-dataset/train/<first_character_of_id>/<image_id>.jpg'
        out_path = os.path.join(out_dir, f'{img_id}.jpg')
        try:
            r = requests.get(url, timeout=10)
            if r.status_code == 200:
                with open(out_path, 'wb') as f:
                    f.write(r.content)
        except:
            print(f'❌ Failed: {img_id}')

In [ ]:
sample_ids = filtered['images'].str.split().dropna().explode().astype(str).sample(10).tolist()
download_images(sample_ids)

  0%|          | 0/10 [00:00<?, ?it/s]

In [ ]:
def extract_descriptors(img_path):
    img = cv2.imread(img_path, 0)
    sift = cv2.SIFT_create()
    kp, des = sift.detectAndCompute(img, None)
    return kp, des, img

def detect_landmark(query_img_path, template_dir="gld_templates"):
    kp1, des1, query_img = extract_descriptors(query_img_path)
    bf = cv2.BFMatcher()
    best_match = None
    max_good = 0

    for file in os.listdir(template_dir):
        if not file.endswith('.jpg'):
            continue
        path = os.path.join(template_dir, file)
        kp2, des2, template = extract_descriptors(path)
        if des1 is None or des2 is None:
            continue
        matches = bf.knnMatch(des1, des2, k=2)
        good = [m for m, n in matches if m.distance < 0.75 * n.distance]
        if len(good) > max_good and len(good) > 10:
            max_good = len(good)
            best_match = (file, kp1, kp2, good, template)

    if best_match:
        file, kp1, kp2, good, template = best_match
        src_pts = np.float32([kp1[m.queryIdx].pt for m in good]).reshape(-1,1,2)
        dst_pts = np.float32([kp2[m.trainIdx].pt for m in good]).reshape(-1,1,2)
        M, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 5.0)
        h, w = template.shape
        corners = np.float32([[0,0],[0,h],[w,h],[w,0]]).reshape(-1,1,2)
        dst = cv2.perspectiveTransform(corners, M)
        out = cv2.imread(query_img_path)
        cv2.polylines(out, [np.int32(dst)], True, (0,255,0), 3)
        tag = file.replace('.jpg','').replace('_',' ').title()
        cv2.putText(out, tag, (30,30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)
        return out
    else:
        print("❌ No match found.")
        return None

In [ ]:
result = detect_landmark("/content/Screenshot 2025-06-18 at 10.31.05 PM.png")

if result is not None:
    from google.colab.patches import cv2_imshow
    cv2_imshow(result)


❌ No match found.


In [ ]:
import pandas as pd

# Load previously filtered landmark images
filtered = pd.read_csv("filtered_landmark_images.csv")

# Make sure we sample valid image IDs as strings
sample_ids = filtered['images'].str.split().dropna().explode().astype(str).sample(10).tolist()
print("✅ Sample image IDs:", sample_ids)

✅ Sample image IDs: ['10366c9b8a19206f', '610377e121bb3508', 'c67157eeaf02f1c9', '9c7de6843524ef20', '4e97a05c03ae2e69', 'ae765332733d5863', '344ebd997f68dffc', '43367bb04b30cfcc', 'ee937d59bbe3471d', 'e4cfb8573c3ad3b3']


In [ ]:
import os
import requests
from tqdm.notebook import tqdm

os.makedirs("gld_templates", exist_ok=True)

def download_images(image_ids, out_dir='gld_templates'):
    # Iterate through each image ID using a progress bar for visual feedback
    for img_id in tqdm(image_ids):
        # Skip invalid or non-string image IDs to prevent errors
        if not img_id or not isinstance(img_id, str):
            continue

        # Build the output path for the current image
        out_path = os.path.join(out_dir, f"{img_id}.jpg")

        # Skip download if the file already exists (avoid duplicates)
        if os.path.exists(out_path):
            continue

        try:
            # Compose the image URL using the image ID (URL pattern may need to be customized)
            url = f"https://example.com/images/{img_id}.jpg"

            # Attempt to download the image from the constructed URL
            r = requests.get(url, timeout=10)

            # Save the image if the request was successful
            if r.status_code == 200:
                with open(out_path, 'wb') as f:
                    f.write(r.content)
            else:
                # Log HTTP errors for debugging and traceability
                print(f"❌ {img_id}: HTTP {r.status_code}")
        except Exception as e:
            # Log any exceptions that occur during the download process
            print(f"❌ {img_id}: {e}")



In [ ]:
download_images(sample_ids)


  0%|          | 0/10 [00:00<?, ?it/s]

❌ 10366c9b8a19206f: HTTP 404
❌ 610377e121bb3508: HTTP 404
❌ c67157eeaf02f1c9: HTTP 404
❌ 9c7de6843524ef20: HTTP 404
❌ 4e97a05c03ae2e69: HTTP 404
❌ ae765332733d5863: HTTP 404
❌ 344ebd997f68dffc: HTTP 404
❌ 43367bb04b30cfcc: HTTP 404
❌ ee937d59bbe3471d: HTTP 404
❌ e4cfb8573c3ad3b3: HTTP 404


In [ ]:
import cv2
import numpy as np
import os
from google.colab.patches import cv2_imshow

def extract_descriptors(image_path):
    img = cv2.imread(image_path, 0)
    sift = cv2.SIFT_create()
    kp, des = sift.detectAndCompute(img, None)
    return kp, des, img

# Example of loading known landmarks
# known_landmarks = [
#     {'tag': 'Eiffel Tower', 'des': descriptors1},
#     {'tag': 'Statue of Liberty', 'des': descriptors2},
#     ...
# ]

def match_landmark(query_img_path, known_landmarks):
    kp_query, des_query, query_img = extract_descriptors(query_img_path)
    bf = cv2.BFMatcher()
    best_match = None
    max_good_matches = 0

    for landmark in known_landmarks:
        matches = bf.knnMatch(des_query, landmark['des'], k=2)
        good = []
        for m, n in matches:
            if m.distance < 0.75 * n.distance:
                good.append(m)
        if len(good) > max_good_matches:
            max_good_matches = len(good)
            best_match = landmark

    output = cv2.cvtColor(query_img, cv2.COLOR_GRAY2BGR)
    if best_match and max_good_matches > 10:
        cv2.putText(
            output,
            best_match['tag'],
            (30, 30),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0,255,0),
            2
        )
        cv2_imshow(output)
    else:
        print("No known landmark matched.")

# Example usage:
# match_landmark('path/to/query.jpg', known_landmarks)